## Split data into Training and Test/holdout 

### 1. Import python libraries

In [1]:
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import train_test_split

### 2. Set options for Jupyter notebook

In [2]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
# Force pandas to display 2 decimal places (no scientific notation)
pd.options.display.float_format = '{:.2f}'.format

### 3. Read the cleaned file

In [3]:
filepath = '../data/interim/neiss_interim_data.parquet'
data_cleaned = pd.read_parquet(filepath)

In [4]:
data_cleaned.info()

<class 'pandas.DataFrame'>
Index: 7314956 entries, 0 to 7315731
Data columns (total 28 columns):
 #   Column                Dtype         
---  ------                -----         
 0   data_year             int64         
 1   CPSC_Case_Number      int64         
 2   Treatment_Date        datetime64[us]
 3   Age                   int64         
 4   Sex                   int64         
 5   Race                  int64         
 6   Other_Race            str           
 7   Hispanic              float64       
 8   Body_Part             int64         
 9   Diagnosis             int64         
 10  Other_Diagnosis       str           
 11  Body_Part_2           float64       
 12  Diagnosis_2           float64       
 13  Other_Diagnosis_2     str           
 14  Disposition           int64         
 15  Location              int64         
 16  Fire_Involvement      int64         
 17  Product_1             str           
 18  Product_2             int64         
 19  Product_3       

### 4. Record Selection 

In [5]:
# Obtain the year from the Treatment_Date column and create a new column called Treatment_Year 
data_cleaned['Treatment_Year'] = data_cleaned['Treatment_Date'].dt.year
# Derive the column "Hospitalized"  

In [6]:
# Define cutoff dates
start_date_train = "2018-01-01"  # training ends
cutoff_date_train = "2021-12-31"  # training ends

cutoff_date_eval = "2023-12-31"   # validation starts
cutoff_date_holdout = "2024-01-01"  # holdout starts

In [7]:
# Train: Betweeen 2018 - 2021 (exclude 2020)
train_df = data_cleaned[(data_cleaned["Treatment_Date"] >= start_date_train) & (data_cleaned["Treatment_Date"] <= cutoff_date_train)]

# Validation/Eval: 2021–2023
eval_df = data_cleaned[(data_cleaned["Treatment_Date"] > cutoff_date_train) & (data_cleaned["Treatment_Date"] < cutoff_date_eval)]

# Holdout: 2024
holdout_df = data_cleaned[data_cleaned["Treatment_Date"] >= cutoff_date_holdout]

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)
print("Holdout shape:", holdout_df.shape)

Train shape: (1370070, 29)
Eval shape: (660851, 29)
Holdout shape: (361645, 29)


In [8]:
train_df.groupby(['Treatment_Year']).size()

Treatment_Year
2018    361635
2019    358681
2020    309343
2021    340411
dtype: int64

In [9]:
#Remove records from train_df where treatment_year =2020
train_df = train_df[train_df['Treatment_Year'] != 2020]

In [10]:
train_df.groupby(['Treatment_Year']).size()

Treatment_Year
2018    361635
2019    358681
2021    340411
dtype: int64

In [11]:
eval_df.groupby(['Treatment_Year']).size()

Treatment_Year
2022    323321
2023    337530
dtype: int64

In [12]:
holdout_df.groupby(['Treatment_Year']).size()

Treatment_Year
2024    361645
dtype: int64

In [13]:
# Remove the treatment year temp column from all three datasets
train_df = train_df.drop(columns=['Treatment_Year'])
eval_df = eval_df.drop(columns=['Treatment_Year'])
holdout_df = holdout_df.drop(columns = ['Treatment_Year'])

In [14]:
print(f'The train data has {train_df.shape[0]} rows and {train_df.shape[1]} features')
print(f'The eval data has {eval_df.shape[0]} rows and {eval_df.shape[1]} features')
print(f'The holdout data has {holdout_df.shape[0]} rows and {holdout_df.shape[1]} features')

The train data has 1060727 rows and 28 features
The eval data has 660851 rows and 28 features
The holdout data has 361645 rows and 28 features


In [15]:
def temporal_stratified_split(df, year_col='stratified_split_key'):
    """
    Splits the dataset into 70% Train, 15% Eval, and 15% Test, 
    ensuring every single year is represented equally in all three datasets.
    """
    # Note: If you only have a full 'Date' column, extract the year first:
    # df['Treatment_Year'] = pd.to_datetime(df['Treatment_Date']).dt.year

    print(f"Total original records: {len(df):,}")

    # STEP 1: Split off the 15% Test Set
    # We stratify based on the Year column
    df_temp, df_test = train_test_split(
        df, 
        test_size=0.15, 
        stratify=df[year_col], 
        random_state=42 # Random state ensures reproducibility
    )

    # STEP 2: Split the remaining 85% into Train (70%) and Eval (15%)
    # Math: 0.15 / 0.85 = 0.17647
    df_train, df_eval = train_test_split(
        df_temp, 
        test_size=(0.15 / 0.85), 
        stratify=df_temp[year_col], 
        random_state=42
    )

    # STEP 3: Verify the distribution
    print("\n--- Split Verification ---")
    print(f"Training Set: {len(df_train):,} rows ({len(df_train)/len(df):.1%})")
    print(f"Eval Set:     {len(df_eval):,} rows ({len(df_eval)/len(df):.1%})")
    print(f"Test Set:     {len(df_test):,} rows ({len(df_test)/len(df):.1%})")
    
    # Prove that the stratification worked (Checking a specific year, e.g., 2020)
    print("\n--- Year Distribution in Train vs Test ---")
    print("Train Years:\n", df_train[year_col].value_counts())
    print("Test Years:\n", df_test[year_col].value_counts())

    return df_train, df_eval, df_test

# Usage:
train_df, eval_df, holdout_df = temporal_stratified_split(data_cleaned, year_col='stratified_split_key')

Total original records: 7,314,956

--- Split Verification ---
Training Set: 5,120,468 rows (70.0%)
Eval Set:     1,097,244 rows (15.0%)
Test Set:     1,097,244 rows (15.0%)

--- Year Distribution in Train vs Test ---
Train Years:
 stratified_split_key
2010-0    263406
2011-0    257017
2012-0    254802
2009-0    254571
2017-0    244791
2008-0    244657
2007-0    242403
2013-0    241922
2006-0    238747
2016-0    238688
2005-0    236967
2014-0    236169
2015-0    229419
2018-0    225755
2019-0    223061
2024-0    221267
2021-0    208390
2023-0    207123
2022-0    198672
2020-0    187255
2024-1     31885
2021-1     29897
2023-1     29635
2020-1     29286
2019-1     28016
2022-1     27652
2018-1     27390
2017-1     26011
2016-1     23927
2015-1     21950
2013-1     21896
2012-1     21240
2014-1     21056
2010-1     20564
2011-1     20500
2009-1     19747
2008-1     17293
2007-1     16439
2006-1     15748
2005-1     15254
Name: count, dtype: int64
Test Years:
 stratified_split_key
2010-0  

In [16]:
train_df['Hospitalized'].value_counts()

Hospitalized
0    4655082
1     465386
Name: count, dtype: int64

In [17]:
data_cleaned['Hospitalized'].value_counts()

Hospitalized
0    6650119
1     664837
Name: count, dtype: int64

### 4. Copy the data into the Processed Folder as a parquet file.

In [18]:
# Copy the training data into the processed folder
train_df.to_parquet('../data/processed/neiss_clean_train.parquet')
# Copy the evaluation data into the processed folder
eval_df.to_parquet('../data/processed/neiss_clean_eval.parquet')
# Copy the holdout dara into the processed folder
holdout_df.to_parquet('../data/processed/neiss_clean_holdout.parquet')

Data saved in parquet file to reduce the size.

Force run the python garbage collector

In [19]:
n = gc.collect()
print(f"Number of unreachable objects collected: {n}")

Number of unreachable objects collected: 0
